### SpatialPeeler benchmark — multi-puck (5 case + 5 control), NMF k=30, cluster-robust SEs

Uses the 10-sample benchmark generated by `B.1.benchmark_posPC2_multiPuck.ipynb`
(pucks 02–06, each split into top = control and bottom = case).

**Why 5 samples per group?**  
Cluster-robust standard errors require ≥ 20–30 clusters to be asymptotically reliable.
With G = 10 total samples this benchmark gives a meaningful validation of the SE correction
that the 2-sample version (`B.2._clusterSE`) could not provide.

**Ground truth:** `obs['in_circle']` on case pucks — beads perturbed with Poisson noise
on the top-100 positive PC2 genes.

In [55]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scanpy as sc
import anndata as ad
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score, roc_curve

warnings.simplefilter('ignore', category=ConvergenceWarning)
sc.settings.verbosity = 1

# SpatialPeeler imports
ROOT = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler')
sys.path.insert(0, str(ROOT))
from SpatialPeeler import case_prediction as cpred
from SpatialPeeler import plotting as plot

RAND_SEED = 28
np.random.seed(RAND_SEED)

## 1. Load benchmark data

In [56]:
DATA_DIR  = ROOT / 'benchmark' / 'generated_benchmark_data'
PUCK_IDS  = ['01', '04', '06', '07']   # pucks with correct orientation

ctrl_list, case_list = [], []

for puck_id in PUCK_IDS:
    ctrl = ad.read_h5ad(DATA_DIR / f'adata_puck{puck_id}_top.h5ad')
    case = ad.read_h5ad(DATA_DIR / f'adata_puck{puck_id}_bot_case_posPC2.h5ad')
    ctrl_list.append(ctrl)
    case_list.append(case)
    print(f'Puck {puck_id}:  ctrl={ctrl.n_obs:,}  case={case.n_obs:,}'
          f'  in_circle={case.obs["in_circle"].sum():,}')

print(f'\nLoaded {len(ctrl_list)} control + {len(case_list)} case pucks  (G = {2*len(PUCK_IDS)} clusters)')

Puck 01:  ctrl=29,753  case=29,750  in_circle=4,463
Puck 04:  ctrl=37,654  case=37,653  in_circle=5,648
Puck 06:  ctrl=39,900  case=39,900  in_circle=5,985
Puck 07:  ctrl=43,046  case=43,037  in_circle=6,456

Loaded 4 control + 4 case pucks  (G = 8 clusters)


In [ ]:
for puck_id, ctrl, case in zip(PUCK_IDS, ctrl_list, case_list):
    ctrl.obs['sample_id'] = f'puck{puck_id}_ctrl'
    ctrl.obs['status']    = 0
    ctrl.obs['Condition'] = 'Control'
    ctrl.obs['in_circle'] = False

    case.obs['sample_id'] = f'puck{puck_id}_case'
    case.obs['status']    = 1
    case.obs['Condition'] = 'Case'

for puck_id, ctrl, case in zip(PUCK_IDS, ctrl_list, case_list):
    print(f'Puck {puck_id}  ctrl in_circle={ctrl.obs["in_circle"].sum()}  '
          f'case in_circle={case.obs["in_circle"].sum():,}')

In [ ]:
top100_pc2 = pd.read_csv(DATA_DIR / 'top100_posPC2_genes.csv')  # positive-PC2 genes
sim_gene_symbols = top100_pc2['gene'].tolist()
print(f'Loaded {len(sim_gene_symbols)} simulation genes: {sim_gene_symbols[:8]} ...')
print(f'All loadings positive: {(top100_pc2["pc2_loading"] > 0).all()}')

## 2. Concatenate and preprocess

In [ ]:
all_samples = ctrl_list + case_list
all_keys    = ([f'puck{p}_ctrl' for p in PUCK_IDS] +
               [f'puck{p}_case' for p in PUCK_IDS])

adata = ad.concat(all_samples, join='inner', merge='first',
                  label='sample_id', keys=all_keys, index_unique='-')

# Restore per-spot columns lost during concat
for col in ['status', 'Condition', 'in_circle']:
    adata.obs[col] = np.concatenate([s.obs[col].values for s in all_samples])

adata.obs['status']    = adata.obs['status'].astype(int)
adata.obs['in_circle'] = adata.obs['in_circle'].astype(bool)

print(adata)
print(adata.obs[['sample_id', 'status', 'in_circle']].value_counts().sort_index())

In [ ]:
adata.layers['counts'] = adata.X.copy()
print(f'Before gene filter: {adata.shape}')

# Always keep sim genes regardless of min_cells
sim_set   = set(sim_gene_symbols)
is_sim    = np.array([g in sim_set for g in adata.var['features'].values])
min_cells = max(1, adata.n_obs // 1000)
n_expr    = np.array((adata.X > 0).sum(axis=0)).flatten()
keep_genes = (n_expr >= min_cells) | is_sim
adata = adata[:, keep_genes].copy()
print(f'Gene filter: {keep_genes.sum():,} kept (min_cells={min_cells})')
print(f'Sim genes after gene filter: {np.array([g in sim_set for g in adata.var["features"].values]).sum()} / {len(sim_gene_symbols)}')

adata.obs['total_counts'] = np.array(adata.X.sum(axis=1)).flatten()
print(f'After filter: {adata.shape}')

In [ ]:
present = set(adata.var['features'].values)
missing_sim_genes = [g for g in sim_gene_symbols if g not in present]
if missing_sim_genes:
    print(f"Warning: {len(missing_sim_genes)} sim genes missing: {missing_sim_genes}")
else:
    print(f"All {len(sim_gene_symbols)} simulation genes present after gene filter.")

In [ ]:
adata.layers['lognorm'] = adata.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4, layer='lognorm')
sc.pp.log1p(adata, layer='lognorm')

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    batch_key='sample_id',
    flavor='seurat',
    layer='lognorm',
    subset=False
)

# Force sim genes into HVG set
sim_set = set(sim_gene_symbols)
is_sim  = np.array([g in sim_set for g in adata.var['features'].values])
adata.var['highly_variable'] = adata.var['highly_variable'] | is_sim
print(f'HVG + sim: {adata.var["highly_variable"].sum()} genes')

adata = adata[:, adata.var['highly_variable']].copy()
sim_kept = np.array([g in sim_set for g in adata.var['features'].values]).sum()
print(f'After HVG subset: {adata.shape},  sim genes: {sim_kept}/{len(sim_gene_symbols)}')

In [ ]:
scale_features = True
if scale_features:
    # unit-variance scaling WITHOUT centering — keeps non-negativity for NMF
    sc.pp.scale(adata, zero_center=False)

## 3. NMF decomposition

In [ ]:
N_FACTORS = 30

X_nmf = adata.X
if sp.issparse(X_nmf):
    X_nmf = X_nmf.toarray()
X_nmf = X_nmf.astype(np.float32)

nmf_model = NMF(
    n_components=N_FACTORS,
    init='nndsvda',
    random_state=RAND_SEED,
    max_iter=1000,
    solver='cd',
)
W = nmf_model.fit_transform(X_nmf)
H = nmf_model.components_

adata.obsm['X_nmf'] = W
adata.uns['nmf_components'] = H

print(f'NMF W: {W.shape},  reconstruction error: {nmf_model.reconstruction_err_:.2f}')

In [ ]:
sample_ids = adata.obs['sample_id'].unique().tolist()
adata_by_sample = {
    sid: adata[adata.obs['sample_id'] == sid].copy()
    for sid in sample_ids
}
n_samples = len(sample_ids)
print(f'{n_samples} samples:', sample_ids)

# Spatial grid: rows = samples, cols = ctrl | case, one figure per factor
for factor_idx in range(N_FACTORS):
    fig, axes = plt.subplots(2, len(PUCK_IDS),
                             figsize=(4 * len(PUCK_IDS), 8),
                             squeeze=False)
    fig.suptitle(f'Factor {factor_idx+1}', fontsize=12)
    for col_i, puck_id in enumerate(PUCK_IDS):
        for row_i, role in enumerate(['ctrl', 'case']):
            sid = f'puck{puck_id}_{role}'
            sub = adata_by_sample[sid]
            x   = sub.obs['Raw_Slideseq_X'].values.astype(float)
            y   = sub.obs['Raw_Slideseq_Y'].values.astype(float)
            scores = sub.obsm['X_nmf'][:, factor_idx]
            ax  = axes[row_i][col_i]
            sc_ = ax.scatter(x, y, c=scores, s=1, cmap='magma_r',
                             linewidths=0, rasterized=True)
            plt.colorbar(sc_, ax=ax, shrink=0.4, aspect=15, pad=0.02)
            ax.set_aspect('equal', 'box')
            ax.axis('off')
            ax.set_title(f'{sid}', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualise sim gene expression across all samples (one figure per gene)
GENES_TO_PLOT = sim_gene_symbols[:3]
feat_to_idx   = {g: i for i, g in enumerate(adata.var['features'].values)}

for gene in GENES_TO_PLOT:
    if gene not in feat_to_idx:
        print(f'{gene} not in adata after HVG subset — skipping')
        continue
    col = feat_to_idx[gene]
    fig, axes = plt.subplots(2, len(PUCK_IDS),
                             figsize=(4 * len(PUCK_IDS), 8),
                             squeeze=False)
    fig.suptitle(gene, fontsize=12)
    for col_i, puck_id in enumerate(PUCK_IDS):
        for row_i, role in enumerate(['ctrl', 'case']):
            sid  = f'puck{puck_id}_{role}'
            sub  = adata[adata.obs['sample_id'] == sid]
            expr = sub.layers['counts'][:, col]
            if sp.issparse(expr):
                expr = expr.toarray().ravel()
            else:
                expr = np.asarray(expr).ravel()
            expr = np.log1p(expr)
            x    = sub.obs['Raw_Slideseq_X'].values.astype(float)
            y    = sub.obs['Raw_Slideseq_Y'].values.astype(float)
            ax   = axes[row_i][col_i]
            sc_  = ax.scatter(x, y, c=expr, s=1, cmap='magma_r',
                              linewidths=0, rasterized=True)
            plt.colorbar(sc_, ax=ax, fraction=0.03, pad=0.02)
            ax.set_aspect('equal', 'box'); ax.axis('off')
            ax.set_title(f'{sid}', fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
nmf_df = pd.DataFrame(
    adata.obsm['X_nmf'],
    columns=[f'Factor {i+1}' for i in range(N_FACTORS)]
)
nmf_df['Condition'] = adata.obs['Condition'].values

nmf_long = nmf_df.melt(id_vars='Condition', var_name='Factor', value_name='Score')

fig, ax = plt.subplots(figsize=(N_FACTORS * 0.6, 5))
sns.violinplot(
    x='Factor', y='Score', hue='Condition', data=nmf_long,
    order=[f'Factor {i+1}' for i in range(N_FACTORS)],
    hue_order=['Control', 'Case'],
    palette={'Control': 'skyblue', 'Case': 'salmon'},
    inner='quartile', cut=0, density_norm='width',
    linewidth=0.6, ax=ax
)
ax.set_xlabel('')
ax.set_ylabel('NMF score', fontsize=11)
ax.set_title('NMF factor score distributions — Control vs Case', fontsize=12)
ax.legend(title='Condition', bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

## 4. Run SpatialPeeler

### Why naive p-values are anti-conservative (everything looks significant)

The standard `sm.Logit` MLE assumes all N spots are independent and identically
distributed (iid) draws. They are not — they are correlated in two nested ways:

**1. Sample-level label clustering**  
Every spot from sample A has `y = 0` (control) and every spot from sample B has `y = 1`
(case). The effective sample size for testing the case/control effect is therefore the
number of *samples*, not the number of spots. Standard errors are deflated by roughly
√(n_spots / n_samples) — a large factor — so even factors that capture nothing
disease-specific will appear highly significant.

**2. Spatial autocorrelation within samples**  
Nearby beads on the same slide share similar expression profiles, adding a second layer of
non-independence on top of the sample-level clustering.


### What cluster-robust SEs fix (and what they don't)

**Cluster-robust SEs (problem 1 — fully addressed)**  
Clustering by `sample_id` uses a sandwich (Huber–White) estimator that allows for
arbitrary within-cluster correlation. The β coefficients and predicted probabilities
(p_hat) are identical to the naive fit — only the standard errors, t-statistics, and
p-values are corrected. In statsmodels this requires passing `cov_type='cluster'` and
`cov_kwds={'groups': sample_ids}` to `.fit()`.

**Spatial autocorrelation within samples (problem 2 — absorbed as nuisance)**  
Clustering at the sample level implicitly absorbs all spatial autocorrelation within each
sample: the sandwich estimator treats the entire within-cluster correlation structure as a
nuisance without modelling it explicitly. This is sufficient here because we are not
trying to estimate spatial ranges or kernels — we just want correct inference on β.


### Why not Newey–West / HAC / spatial HAC?

A natural question is whether Newey–West or Conley (1999) spatial HAC standard errors
would be more principled. The short answer is: **might not be appropiate here**, for two reasons.

**1. Wrong tool for 2D spatial data**  
Standard Newey–West / HAC assumes a **1D ordering** (time series — observations close in
*time* are downweighted by a kernel). For spatial transcriptomics the autocorrelation is
in 2D space, not along a single axis. Using `cov_type='HAC'` on raw spots would require
first sorting beads by some 1D proxy (e.g. x-coordinate), which is an arbitrary and poor
approximation of 2D spatial structure.  
The correct spatial analogue is **Conley (1999) spatial HAC**, which weights score
outer-products by a 2D distance kernel — but this is not in statsmodels and would require
a custom implementation.

**2. Slow**  
Spatial HAC computes a kernel-weighted sum over all pairs of spots within a bandwidth:

```
M_spatial = Σᵢ Σⱼ K(dᵢⱼ / h) · sᵢ sⱼᵀ
```

With n ≈ 80 k spots per sample and a spatial bandwidth, this is O(n × k) at minimum
(with k-NN acceleration) — roughly 100× slower than cluster-robust, which just sums
scores at the sample level in a single pass.

**Conclusion:** for a multi-sample design, clustering by `sample_id` is both the
statistically correct and computationally efficient choice. Spatial HAC would only add
value in a **single-sample** setting where there is no higher-level cluster to group by.

In [ ]:
# Method 1: all spots, no zero-filtering
# Run naive (iid) first for comparison, then cluster-robust (used downstream)
results_v0_naive = cpred.single_factor_logistic_evaluation(
    adata, factor_key='X_nmf', max_factors=N_FACTORS, use_cluster_se=False
)
results_v0 = cpred.single_factor_logistic_evaluation(
    adata, factor_key='X_nmf', max_factors=N_FACTORS, use_cluster_se=True
)

naive_df = pd.DataFrame({
    'factor':     [f'Factor_{r["factor_index"]+1}' for r in results_v0_naive],
    'factor_idx': [r['factor_index'] for r in results_v0_naive],
    'coef':       [r['coef'] for r in results_v0_naive],
    'pval_naive': [r['pval'] for r in results_v0_naive],
})
clust_df = pd.DataFrame({
    'factor':     [f'Factor_{r["factor_index"]+1}' for r in results_v0],
    'factor_idx': [r['factor_index'] for r in results_v0],
    'coef':       [r['coef'] for r in results_v0],
    'pval_clust': [r['pval'] for r in results_v0],
})
coef_df_v0 = naive_df.merge(clust_df[['factor', 'pval_clust']], on='factor') \
                     .sort_values('coef', ascending=False)

print(coef_df_v0.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=False)
colors = ['#e84545' if c > 0 else '#4575b4' for c in coef_df_v0['coef']]
for ax, col, title in zip(axes,
                           ['pval_naive', 'pval_clust'],
                           ['Naive p-value', 'Cluster-robust p-value']):
    ax.bar(coef_df_v0['factor'], -np.log10(coef_df_v0[col].clip(1e-300)), color=colors)
    ax.axhline(-np.log10(0.05), color='black', linewidth=0.8, linestyle='--', label='p=0.05')
    ax.set_xlabel('Factor', fontsize=10)
    ax.set_ylabel('-log10(p-value)', fontsize=11)
    ax.set_title(f'Method 1 (all spots) — {title}', fontsize=12)
    ax.legend(fontsize=9)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Method 2: zero-filtered (active spots only), cluster-robust SEs
import statsmodels.api as sm
from scipy.special import logit

X = adata.obsm['X_nmf']
y = adata.obs['status'].values        # 0 = Control, 1 = Case
sample_labels = adata.obs['sample_id'].values

THRESHOLD = 0.0
VISUALIZE  = True

all_results = []

for i in range(min(N_FACTORS, X.shape[1])):
    Xi        = X[:, i].reshape(-1, 1)
    high_mask = Xi.ravel() > THRESHOLD
    high_idx  = np.where(high_mask)[0]

    if len(high_idx) == 0:
        print(f'Factor {i+1}: no active spots — skipping.')
        continue
    if len(np.unique(y[high_idx])) < 2:
        print(f'Factor {i+1}: only one class in active spots — skipping.')
        continue

    Xi_high      = Xi[high_idx]
    y_high       = y[high_idx]
    groups_high  = sample_labels[high_idx]   # cluster IDs for sandwich estimator

    X_int = sm.add_constant(Xi_high)

    # Naive fit (iid) — kept only for p-value comparison
    fit_naive = sm.Logit(y_high, X_int).fit(disp=False)

    # Cluster-robust fit — used for all downstream results
    fit = sm.Logit(y_high, X_int).fit(
        cov_type='cluster',
        cov_kwds={'groups': groups_high},
        disp=False
    )

    p_hat = fit.predict(X_int)   # predicted probabilities are identical for both fits

    all_results.append({
        'factor_index': i,
        'coef':         float(fit.params[1]),
        'intercept':    float(fit.params[0]),
        'std_err':      float(fit.bse[1]),
        'pval':         float(fit.pvalues[1]),         # cluster-robust p-value
        'pval_naive':   float(fit_naive.pvalues[1]),   # naive p-value (for comparison)
        'p_hat':        p_hat,
        'logit_p_hat':  logit(p_hat),
        'high_idx':     high_idx,
        'y_high':       y_high,
    })

    if VISUALIZE:
        df_vis = pd.DataFrame({
            'Condition': adata.obs['Condition'].values[high_idx],
            'p_hat':     p_hat
        })
        from matplotlib.lines import Line2D
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        cond_colors = ['skyblue' if c == 0 else 'salmon' for c in y_high]
        axes[0].scatter(Xi_high.ravel(), p_hat, c=cond_colors,
                        s=3, alpha=0.4, linewidths=0, rasterized=True)
        axes[0].set_xlabel(f'NMF Factor {i+1} score', fontsize=9)
        axes[0].set_ylabel('p_hat', fontsize=9)
        axes[0].set_title(f'Factor {i+1} — NMF vs p_hat', fontsize=10)
        axes[0].legend(handles=[
            Line2D([0],[0], marker='o', color='w', markerfacecolor='skyblue', label='Control'),
            Line2D([0],[0], marker='o', color='w', markerfacecolor='salmon',  label='Case')],
            fontsize=8, frameon=False)
        sns.violinplot(x='Condition', y='p_hat', data=df_vis,
                       order=['Control', 'Case'],
                       palette={'Control': 'skyblue', 'Case': 'salmon'},
                       inner='box', cut=0, ax=axes[1])
        axes[1].set_ylim(0, 1)
        axes[1].set_title(f'Factor {i+1} — p_hat by condition', fontsize=10)
        plt.tight_layout(); plt.show()

results           = all_results
results_by_factor = {r['factor_index']: r for r in results}
print(f'Done. {len(results)} factors evaluated.')

In [ ]:
coef_df = pd.DataFrame({
    'factor':     [f'Factor_{r["factor_index"]+1}' for r in results],
    'factor_idx': [r['factor_index'] for r in results],
    'coef':       [r['coef'] for r in results],
    'pval':       [r['pval'] for r in results],
}).sort_values('coef', ascending=False)

print(coef_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(15, 5))
colors = ['#e84545' if c > 0 else '#4575b4' for c in coef_df['coef']]
ax.bar(coef_df['factor'], coef_df['coef'], color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Factor', fontsize=11)
ax.set_ylabel('Logistic regression coefficient', fontsize=11)
ax.set_title('SpatialPeeler (zero-filtered): factor coefficients', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout(); plt.show()

## 4b. Naive vs cluster-robust p-values — side-by-side comparison

The β coefficients and p_hat scores are identical in both fits. Only the standard errors
(and therefore p-values) differ.

With **G = 10 clusters** (5 case + 5 control pucks) the sandwich estimator is in a
reasonable asymptotic regime — p-values below the diagonal (cluster-robust less
significant than naive) indicate that the iid assumption was indeed anti-conservative, as
expected.

In [ ]:
cmp_df = pd.DataFrame({
    'factor':       [f'Factor_{r["factor_index"]+1}' for r in results],
    'factor_idx':   [r['factor_index'] for r in results],
    'coef':         [r['coef'] for r in results],
    'pval_naive':   [r['pval_naive'] for r in results],
    'pval_clust':   [r['pval'] for r in results],
}).sort_values('factor_idx')

print(cmp_df[['factor', 'coef', 'pval_naive', 'pval_clust']].to_string(index=False))

# ── Panel 1: bar chart comparing -log10(p) for each factor ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=False)
colors = ['#e84545' if c > 0 else '#4575b4' for c in cmp_df['coef']]
for ax, col, title in zip(axes,
                           ['pval_naive', 'pval_clust'],
                           ['Naive (iid)', 'Cluster-robust']):
    ax.bar(cmp_df['factor'], -np.log10(cmp_df[col].clip(1e-300)), color=colors)
    ax.axhline(-np.log10(0.05), color='black', linewidth=0.8, linestyle='--', label='p=0.05')
    ax.set_xlabel('Factor', fontsize=10)
    ax.set_ylabel('-log10(p-value)', fontsize=11)
    ax.set_title(f'Method 2 (zero-filtered) — {title}', fontsize=12)
    ax.legend(fontsize=9)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

# ── Panel 2: scatter naive vs cluster-robust ─────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))
lp_naive = -np.log10(cmp_df['pval_naive'].clip(1e-300))
lp_clust = -np.log10(cmp_df['pval_clust'].clip(1e-300))
ax.scatter(lp_naive, lp_clust, c=colors, s=60, edgecolors='white', linewidths=0.4)
lim = max(lp_naive.max(), lp_clust.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', lw=0.8, label='y = x')
ax.axvline(-np.log10(0.05), color='grey', lw=0.6, linestyle=':')
ax.axhline(-np.log10(0.05), color='grey', lw=0.6, linestyle=':')
ax.set_xlabel('Naive  −log10(p)', fontsize=11)
ax.set_ylabel('Cluster-robust  −log10(p)', fontsize=11)
ax.set_title('p-value inflation under iid assumption\n(points above diagonal = over-significant in naive)',
             fontsize=10)
ax.legend(fontsize=9)

# Annotate the best factor
best_row = cmp_df.loc[cmp_df['coef'].abs().idxmax()]
ax.annotate(best_row['factor'],
            xy=(float(-np.log10(best_row['pval_naive'] + 1e-300)),
                float(-np.log10(best_row['pval_clust'] + 1e-300))),
            xytext=(8, -12), textcoords='offset points', fontsize=8)
plt.tight_layout()
plt.show()

sig_naive = (cmp_df['pval_naive'] < 0.05).sum()
sig_clust = (cmp_df['pval_clust'] < 0.05).sum()
print(f'Factors p<0.05 — naive: {sig_naive},  cluster-robust: {sig_clust}')

## 5. Spatial visualisation of p_hat

In [ ]:
def _full_p_hat(r, n):
    ph = np.full(n, np.nan)
    ph[r['high_idx']] = r['p_hat']
    return ph

top_factors = coef_df[coef_df['coef'] > 0].head(5)['factor_idx'].tolist()
print('Top GOF factors:', [f+1 for f in top_factors])

for fi in top_factors:
    if fi not in results_by_factor:
        continue
    r    = results_by_factor[fi]
    ph   = _full_p_hat(r, adata.n_obs)
    coef_val = coef_df.loc[coef_df['factor_idx'] == fi, 'coef'].values[0]

    fig, axes = plt.subplots(2, len(PUCK_IDS),
                             figsize=(4 * len(PUCK_IDS), 8),
                             squeeze=False)
    fig.suptitle(f'Factor {fi+1}  (coef={coef_val:.3f})', fontsize=11)

    for col_i, puck_id in enumerate(PUCK_IDS):
        for row_i, role in enumerate(['ctrl', 'case']):
            sid     = f'puck{puck_id}_{role}'
            sub     = adata[adata.obs['sample_id'] == sid]
            sub_idx = np.where(adata.obs['sample_id'] == sid)[0]
            x       = sub.obs['Raw_Slideseq_X'].values.astype(float)
            y_coord = sub.obs['Raw_Slideseq_Y'].values.astype(float)
            ph_sub  = ph[sub_idx]
            active  = ~np.isnan(ph_sub)
            ax      = axes[row_i][col_i]

            ax.scatter(x[~active], y_coord[~active], c='#e8e8e8',
                       s=0.3, linewidths=0, rasterized=True)
            sc_ = ax.scatter(x[active], y_coord[active], c=ph_sub[active],
                             s=1, cmap='YlGnBu', linewidths=0, rasterized=True)
            plt.colorbar(sc_, ax=ax, shrink=0.4, aspect=15, pad=0.02)
            ax.set_aspect('equal', 'box'); ax.axis('off')
            ax.set_title(f'{sid}  ({active.sum():,})', fontsize=8)

    plt.tight_layout(); plt.show()

## 6. Evaluate recovery of the perturbed circle

For each factor, compute AUROC between p_hat (on case beads only) and the ground-truth `in_circle` label.  
A high AUROC means the factor's p_hat distinguishes perturbed from unperturbed case beads.

In [ ]:
case_global_idx = np.where(adata.obs['status'].values == 1)[0]
gt = adata.obs['in_circle'].values[case_global_idx].astype(int)

def _case_p_hat(r, n, case_idx):
    """p_hat for case beads; 0 for inactive case beads (zero-NMF)."""
    ph = np.zeros(n)
    ph[r['high_idx']] = r['p_hat']
    return ph[case_idx]

auroc_rows = []
for r in results:
    ph_case = _case_p_hat(r, adata.n_obs, case_global_idx)
    try:
        auc = roc_auc_score(gt, ph_case)
    except Exception:
        auc = np.nan
    auroc_rows.append({
        'factor':     f'Factor_{r["factor_index"]+1}',
        'factor_idx': r['factor_index'],
        'coef':       r['coef'],
        'auroc':      auc
    })

auroc_df = pd.DataFrame(auroc_rows).sort_values('auroc', ascending=False)
print(auroc_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(17, 4))
ax.bar(auroc_df['factor'], auroc_df['auroc'],
       color=['#e84545' if a >= 0.7 else '#aaaaaa' for a in auroc_df['auroc']])
ax.axhline(0.5, color='black', linewidth=0.8, linestyle='--', label='random')
ax.set_ylim(0, 1)
ax.set_xlabel('Factor')
ax.set_ylabel('AUROC (p_hat vs in_circle, case beads only)')
ax.set_title('Circle recovery per factor')
ax.legend(fontsize=9)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
best_fi  = int(auroc_df.iloc[0]['factor_idx'])
best_auc = auroc_df.iloc[0]['auroc']
best_ph  = _case_p_hat(results_by_factor[best_fi], adata.n_obs, case_global_idx)

fpr, tpr, _ = roc_curve(gt, best_ph)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color='#e84545', lw=2,
        label=f'Factor {best_fi+1}  (AUC={best_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC — best factor (p_hat vs in_circle, case beads)')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Show p_hat vs ground truth for the best factor, one panel per case puck
case_sids = [f'puck{p}_case' for p in PUCK_IDS]
ph_full   = _full_p_hat(results_by_factor[best_fi], adata.n_obs)

fig, axes = plt.subplots(2, len(PUCK_IDS),
                         figsize=(4 * len(PUCK_IDS), 8),
                         squeeze=False)
fig.suptitle(f'Best factor: Factor {best_fi+1}  (AUROC={best_auc:.3f})', fontsize=11)

for col_i, (puck_id, sid) in enumerate(zip(PUCK_IDS, case_sids)):
    sub      = adata[adata.obs['sample_id'] == sid]
    sub_idx  = np.where(adata.obs['sample_id'] == sid)[0]
    ph_sub   = ph_full[sub_idx]
    active   = ~np.isnan(ph_sub)
    gt_bool  = sub.obs['in_circle'].values.astype(bool)
    x        = sub.obs['Raw_Slideseq_X'].values.astype(float)
    y        = sub.obs['Raw_Slideseq_Y'].values.astype(float)

    # Row 0: p_hat
    ax0 = axes[0][col_i]
    ax0.scatter(x[~active], y[~active], c='#e8e8e8', s=0.3, linewidths=0, rasterized=True)
    sc_ = ax0.scatter(x[active], y[active], c=ph_sub[active],
                      s=0.8, cmap='magma_r', vmin=0, vmax=1,
                      linewidths=0, rasterized=True)
    plt.colorbar(sc_, ax=ax0, fraction=0.03, pad=0.02, label='p_hat')
    ax0.set_title(f'p_hat  {sid}', fontsize=8)
    ax0.set_aspect('equal', 'box'); ax0.axis('off')

    # Row 1: ground truth
    ax1 = axes[1][col_i]
    colors = np.where(gt_bool, '#e84545', '#d3d3d3')
    ax1.scatter(x, y, c=colors, s=0.5, linewidths=0, rasterized=True)
    ax1.set_title(f'ground truth  {sid}', fontsize=8)
    ax1.set_aspect('equal', 'box'); ax1.axis('off')

plt.tight_layout(); plt.show()

## 7. Spot clustering and DE analysis

For each top GOF factor: cluster active spots into `case_0/1` (KMeans on case p_hat) and `control_0/1` (split by the same threshold), then run Wilcoxon DE between every pair of clusters.

In [ ]:
from sklearn.cluster import KMeans

CASE_COND_NAME  = 'Case'
FDR_THRESH      = 0.05
LOGFC_THRESH    = 0.1

var_to_symbol = dict(zip(adata.var_names, adata.var['features']))

def get_num_sig_de(de_df, fdr=0.05, logfc=0.1):
    return ((de_df['pvals_adj'] < fdr) & (de_df['logfoldchanges'].abs() > logfc)).sum()

def run_wilcoxon(adata_sub, grp_mask, ref_mask, grp_label, ref_label):
    keep = grp_mask | ref_mask
    ad   = adata_sub[keep].copy()
    ad.obs['_grp'] = pd.Categorical(
        np.where(grp_mask[keep], grp_label, ref_label),
        categories=[ref_label, grp_label]
    )
    sc.tl.rank_genes_groups(
        ad, groupby='_grp', groups=[grp_label], reference=ref_label,
        method='wilcoxon', corr_method='benjamini-hochberg',
        use_raw=False, layer='lognorm', n_genes=ad.n_vars
    )
    de = sc.get.rank_genes_groups_df(ad, group=grp_label).rename(columns={'names': 'gene'})
    de['gene_name'] = de['gene'].map(var_to_symbol)
    return de

top_gof_idx = coef_df[coef_df['coef'] > 0]['factor_idx'].tolist()[:3]
print(f'Processing {len(top_gof_idx)} GOF factors: {[i+1 for i in top_gof_idx]}')

cluster_mask_dict          = {}
de_results_dict_factorwise = {}

CLUSTER_PALETTE = {
    'control_0': '#54A24B',
    'control_1': '#E45756',
    'case_0':    '#4C78A8',
    'case_1':    '#F58518',
}
CAT_ORDER = ['control_0', 'control_1', 'case_0', 'case_1']

for factor_idx in top_gof_idx:
    if factor_idx not in results_by_factor:
        continue

    print(f"\n{'='*55}\nFactor {factor_idx+1}")
    result   = results_by_factor[factor_idx]
    high_idx = result['high_idx']
    p_hat    = result['p_hat']

    adata_sub = adata[high_idx].copy()
    adata_sub.obs['p_hat'] = p_hat

    mask_case  = adata_sub.obs['Condition'].values == CASE_COND_NAME
    p_hat_case = p_hat[mask_case]

    km       = KMeans(n_clusters=2, random_state=RAND_SEED, n_init='auto')
    km.fit(p_hat_case.reshape(-1, 1))
    centers  = km.cluster_centers_.ravel()
    order    = np.argsort(centers)
    remap    = {order[0]: 0, order[1]: 1}
    case_km  = np.vectorize(remap.get)(km.labels_)
    threshold = centers.mean()
    print(f"  KMeans centroids: {centers.round(3)},  threshold: {threshold:.3f}")

    obs_col        = f'cluster_f{factor_idx+1}'
    cluster_labels = np.empty(len(adata_sub), dtype=object)
    case_local = np.where(mask_case)[0]
    ctrl_local = np.where(~mask_case)[0]
    cluster_labels[case_local] = np.where(case_km == 1, 'case_1', 'case_0')
    cluster_labels[ctrl_local] = np.where(p_hat[~mask_case] >= threshold, 'control_1', 'control_0')

    adata_sub.obs[obs_col] = pd.Categorical(cluster_labels, categories=CAT_ORDER)
    print(adata_sub.obs[obs_col].value_counts().reindex(CAT_ORDER).to_string())
    cluster_mask_dict[obs_col] = adata_sub.obs[obs_col].copy()

    df_vio = pd.DataFrame({'cluster': adata_sub.obs[obs_col].values, 'p_hat': p_hat})
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.violinplot(x='cluster', y='p_hat', data=df_vio, order=CAT_ORDER,
                   palette=CLUSTER_PALETTE, inner='box', cut=0, ax=ax)
    ax.set_ylim(0, 1); ax.set_xlabel('')
    ax.set_title(f'Factor {factor_idx+1} — p_hat by cluster', fontsize=11)
    plt.tight_layout(); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, sid in zip(axes, sample_ids):
        sub     = adata_sub[adata_sub.obs['sample_id'] == sid]
        x       = sub.obs['Raw_Slideseq_X'].values.astype(float)
        y_coord = sub.obs['Raw_Slideseq_Y'].values.astype(float)
        colors  = [CLUSTER_PALETTE[c] for c in sub.obs[obs_col]]
        ax.scatter(x, y_coord, c=colors, s=0.5, linewidths=0, rasterized=True)
        ax.set_aspect('equal', 'box'); ax.axis('off')
        ax.set_title(f'Factor {factor_idx+1} — {sid}', fontsize=9)
    handles = [mpatches.Patch(color=CLUSTER_PALETTE[k], label=k) for k in CAT_ORDER]
    axes[1].legend(handles=handles, fontsize=8, frameon=False, loc='lower right')
    plt.tight_layout(); plt.show()

    obs = adata_sub.obs[obs_col].values
    m   = {k: obs == k for k in CAT_ORDER}

    comparisons = [
        ('case1_vs_case0',       m['case_1'],    m['case_0']),
        ('case1_vs_other',       m['case_1'],    ~m['case_1']),
        ('case1_vs_control1',    m['case_1'],    m['control_1']),
        ('case0_vs_control',     m['case_0'],    m['control_0'] | m['control_1']),
        ('case0_vs_control0',    m['case_0'],    m['control_0']),
        ('control1_vs_control0', m['control_1'], m['control_0']),
    ]

    de_results_dict = {}
    num_sig_DE      = {}

    for comp_name, grp_mask, ref_mask in comparisons:
        grp_label, ref_label = comp_name.split('_vs_')
        if grp_mask.sum() < 5 or ref_mask.sum() < 5:
            print(f"  {comp_name}: too few spots — skipped")
            continue
        de = run_wilcoxon(adata_sub, grp_mask, ref_mask, grp_label, ref_label)
        n  = get_num_sig_de(de, FDR_THRESH, LOGFC_THRESH)
        print(f"  {comp_name}: {n} sig DE genes")
        de_results_dict[comp_name] = de
        num_sig_DE[comp_name]      = n

    de_results_dict_factorwise[f'factor_{factor_idx+1}'] = de_results_dict

    if num_sig_DE:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(list(num_sig_DE.keys()), list(num_sig_DE.values()),
               color=sns.color_palette('viridis', len(num_sig_DE)))
        ax.set_ylabel('# sig DE genes')
        ax.set_title(f'Factor {factor_idx+1} — significant DE genes per comparison', fontsize=11)
        plt.xticks(rotation=40, ha='right', fontsize=9)
        plt.tight_layout(); plt.show()

print('\nDone.')